# Experiment 1: Single Best Models (Baseline)

## Behaviour 
Logistic Regression\
Recall = 0.85\
F1 = 0.80

## Academic 
Random Forest\
Recall = 1.00\
F1 = 0.889


In [9]:
# check behaviour and academic validation sets aligned 
import pandas as pd

df_behaviour = pd.read_csv("../datasets/X_beh_val.csv")
df_academic = pd.read_csv("../datasets/X_aca_val.csv")

# Experiment 2:  Weighted Ensemble 
Behaviour probability = Pb\
Academic probability = Pa\

Average: ensemble_prob = (
    Pb +
    Pa
) / 2

Weighted combination: \
0.6 Academic\
0.4 Behaviour\
\
0.7 Academic\
0.3 Behaviour\
\
0.8 Academic\
0.2 Behaviour

## Objective

Combine the probability outputs from the Behaviour Risk Model and Academic Risk Model into a single student risk probability.

Unlike voting, this approach combines the predicted probabilities, which is much more appropriate for dashboard because the dashboard ultimately needs to display a risk score (%).

In [13]:
# load both final models 
import joblib
from pathlib import Path

# robust file path handling
model_dir = Path.cwd().parent / "models"

behaviour_model = joblib.load(model_dir / "behaviour_model_v1.pkl")
academic_model = joblib.load(model_dir / "academic_model_v1.pkl")

In [14]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

In [17]:
# load validation datasets
dataset_dir = Path.cwd().parent / "datasets"

X_beh_val = pd.read_csv(dataset_dir / "X_beh_val.csv")
X_aca_val = pd.read_csv(dataset_dir / "X_aca_val.csv")

y_beh_val = pd.read_csv(dataset_dir / "y_beh_val.csv").squeeze()
y_aca_val = pd.read_csv(dataset_dir / "y_aca_val.csv").squeeze()

meta_beh_val = pd.read_csv(dataset_dir / "meta_beh_val.csv")
meta_aca_val = pd.read_csv(dataset_dir / "meta_aca_val.csv")

In [19]:
# load the behaviour scaler
scaler = joblib.load(model_dir / "behaviour_scaler_v1.pkl")
X_beh_val_scaled = scaler.transform(X_beh_val)

In [22]:
# predict probabilities 
Pb = behaviour_model.predict_proba(X_beh_val_scaled)[:, 1]

Pa = academic_model.predict_proba(X_aca_val)[:, 1]

In [21]:
meta_beh_val[
    ["student_key", "course_key", "academic_period_key"]
].equals(
    meta_aca_val[
        ["student_key", "course_key", "academic_period_key"]
    ]
)

True

In [23]:
beh_probs = meta_beh_val.copy()
beh_probs["behaviour_prob"] = Pb

aca_probs = meta_aca_val.copy()
aca_probs["academic_prob"] = Pa
aca_probs["actual"] = y_aca_val.values

In [24]:
ensemble_df = beh_probs.merge(
    aca_probs,
    on=["student_key", "course_key", "academic_period_key", "snapshot_date"],
    how="inner"
)

ensemble_df.shape

(34, 7)

In [28]:
# evaluate ensemble model with different weights
weights = [
    (0.8, 0.2),  # more behaviour
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),  # equal
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8)   # more academic
]

results = []

for w_beh, w_aca in weights:
    ensemble_prob = (
        w_beh * ensemble_df["behaviour_prob"]
        + w_aca * ensemble_df["academic_prob"]
    )

    ensemble_pred = (ensemble_prob >= 0.5).astype(int)
    y_true = ensemble_df["actual"]

    results.append({
        "model": f"Weighted Ensemble B{w_beh}_A{w_aca}",
        "behaviour_weight": w_beh,
        "academic_weight": w_aca,
        "accuracy": accuracy_score(y_true, ensemble_pred),
        "precision": precision_score(y_true, ensemble_pred, zero_division=0),
        "recall": recall_score(y_true, ensemble_pred, zero_division=0),
        "f1": f1_score(y_true, ensemble_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, ensemble_prob)
    })

weighted_results_df = pd.DataFrame(results)
weighted_results_df


,model,behaviour_weight,academic_weight,accuracy,precision,recall,f1,roc_auc
0,Weighted Ensemble B0.8_A0.2,0.8,0.2,0.852941,0.8,1.0,0.888889,0.939286
1,Weighted Ensemble B0.7_A0.3,0.7,0.3,0.852941,0.8,1.0,0.888889,0.939286
2,Weighted Ensemble B0.6_A0.4,0.6,0.4,0.852941,0.8,1.0,0.888889,0.935714
3,Weighted Ensemble B0.5_A0.5,0.5,0.5,0.852941,0.8,1.0,0.888889,0.935714
4,Weighted Ensemble B0.4_A0.6,0.4,0.6,0.852941,0.8,1.0,0.888889,0.921429
5,Weighted Ensemble B0.3_A0.7,0.3,0.7,0.852941,0.8,1.0,0.888889,0.921429
6,Weighted Ensemble B0.2_A0.8,0.2,0.8,0.852941,0.8,1.0,0.888889,0.910714


In [27]:
from sklearn.metrics import classification_report

best_row = weighted_results_df.sort_values(
    by=["recall", "f1", "roc_auc"],
    ascending=False
).iloc[0]

best_w_beh = best_row["behaviour_weight"]
best_w_aca = best_row["academic_weight"]

best_prob = (
    best_w_beh * ensemble_df["behaviour_prob"]
    + best_w_aca * ensemble_df["academic_prob"]
)

best_pred = (best_prob >= 0.5).astype(int)

print(confusion_matrix(ensemble_df["actual"], best_pred))
print(classification_report(ensemble_df["actual"], best_pred, zero_division=0))

[[ 9  5]
 [ 0 20]]
              precision    recall  f1-score   support

           0       1.00      0.64      0.78        14
           1       0.80      1.00      0.89        20

    accuracy                           0.85        34
   macro avg       0.90      0.82      0.84        34
weighted avg       0.88      0.85      0.85        34



In [29]:
ensemble_df[["behaviour_prob", "academic_prob"]].corr()

,behaviour_prob,academic_prob
behaviour_prob,1.000000,0.678373
academic_prob,0.678373,1.000000


In [47]:
# print intercept and coef for behaviour
print("Behaviour Model Intercept:", behaviour_model.intercept_)
print("Behaviour Model Coefficients:", behaviour_model.coef_)

Behaviour Model Intercept: [0.49209318]
Behaviour Model Coefficients: [[-0.06290565 -0.39949096 -0.27624108  0.08045074  0.26372426  0.17082166
   0.3672414   0.16348534 -0.00966057 -0.33951133  0.08745861 -0.11196134
   0.18288784]]


# Experiment 3: Stacking
Inputs: Behaviour probability, Academic probability \
Meta model: Logistic Regression

Behaviour LR + Academic RF = Meta LR

In [31]:
# generate meta-features for meta-model training
meta_X_train = pd.DataFrame({
    "behaviour_prob": ensemble_df["behaviour_prob"],
    "academic_prob": ensemble_df["academic_prob"]
})

meta_y_train = ensemble_df["actual"]

In [32]:
# train meta-model
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    random_state=42
)

meta_model.fit(
    meta_X_train,
    meta_y_train
)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [33]:
# predict probabilities for meta-model
meta_probs = meta_model.predict_proba(meta_X_train)[:, 1]

meta_pred = (meta_probs >= 0.5).astype(int)

In [40]:
# print probabilities and predictions for the first 10 samples
print("Meta-model probabilities:", meta_probs[:10])
print("Meta-model predictions:", meta_pred[:10])


Meta-model probabilities: [0.43794046 0.60101696 0.70706799 0.72892442 0.63025783 0.24587289
 0.61855683 0.7321023  0.79783624 0.28718852]
Meta-model predictions: [0 1 1 1 1 0 1 1 1 0]


In [42]:
# probabilities table
probabilities_table = pd.DataFrame({
    "student_key": ensemble_df["student_key"],
    "course_key": ensemble_df["course_key"],
    "academic_period_key": ensemble_df["academic_period_key"],
    "snapshot_date": ensemble_df["snapshot_date"],
    "behaviour_prob": ensemble_df["behaviour_prob"],
    "academic_prob": ensemble_df["academic_prob"],
    "meta_model_prob": meta_probs,
    "meta_model_pred": meta_pred,
    "actual": ensemble_df["actual"]
})
probabilities_table.head(10)

,student_key,course_key,academic_period_key,snapshot_date,behaviour_prob,academic_prob,meta_model_prob,meta_model_pred,actual
0,39,1,1,2026-06-29,0.405872,0.44,0.437940,0,0
1,74,1,1,2026-06-29,0.614206,0.66,0.601017,1,0
2,130,1,1,2026-06-29,0.663134,0.88,0.707068,1,1
3,148,1,1,2026-06-29,0.690932,0.92,0.728924,1,1
4,21,1,1,2026-06-29,0.495559,0.80,0.630258,1,0
5,82,1,1,2026-06-29,0.368927,0.00,0.245873,0,0
6,138,1,1,2026-06-29,0.788602,0.59,0.618557,1,1
7,8,1,1,2026-06-29,0.800468,0.86,0.732102,1,1
8,170,1,1,2026-06-29,0.888895,1.00,0.797836,1,1
9,142,1,1,2026-06-29,0.500727,0.03,0.287189,0,0


In [38]:
# meta-model on training data 
print(confusion_matrix(meta_y_train, meta_pred))
print("Meta-model accuracy:", accuracy_score(meta_y_train, meta_pred))
print("Meta-model precision:", precision_score(meta_y_train, meta_pred, zero_division=0))
print("Meta-model recall:", recall_score(meta_y_train, meta_pred, zero_division=0))
print("Meta-model F1 score:", f1_score(meta_y_train, meta_pred, zero_division=0))
print("Meta-model ROC AUC score:", roc_auc_score(meta_y_train, meta_probs))
print(classification_report(meta_y_train, meta_pred, zero_division=0))
print("Meta-model coefficients:", meta_model.coef_)
print("Meta-model intercept:", meta_model.intercept_)
print("Meta-model weights for behaviour and academic probabilities:", meta_model.coef_[0])


[[ 9  5]
 [ 0 20]]
Meta-model accuracy: 0.8529411764705882
Meta-model precision: 0.8
Meta-model recall: 1.0
Meta-model F1 score: 0.8888888888888888
Meta-model ROC AUC score: 0.9214285714285714
              precision    recall  f1-score   support

           0       1.00      0.64      0.78        14
           1       0.80      1.00      0.89        20

    accuracy                           0.85        34
   macro avg       0.90      0.82      0.84        34
weighted avg       0.88      0.85      0.85        34

Meta-model coefficients: [[1.17780096 1.88115342]]
Meta-model intercept: [-1.55526901]
Meta-model weights for behaviour and academic probabilities: [1.17780096 1.88115342]


In [36]:
# load test datasets
X_beh_test = pd.read_csv(dataset_dir / "X_beh_test.csv")
X_aca_test = pd.read_csv(dataset_dir / "X_aca_test.csv")

y_beh_test = pd.read_csv(dataset_dir / "y_beh_test.csv").squeeze()
y_aca_test = pd.read_csv(dataset_dir / "y_aca_test.csv").squeeze()

meta_beh_test = pd.read_csv(dataset_dir / "meta_beh_test.csv")
meta_aca_test = pd.read_csv(dataset_dir / "meta_aca_test.csv")

In [37]:
# evaluate meta-model on test set
X_beh_test_scaled = scaler.transform(X_beh_test)

Pb_test = behaviour_model.predict_proba(X_beh_test_scaled)[:, 1]
Pa_test = academic_model.predict_proba(X_aca_test)[:, 1]

# align using test metadata
beh_test_probs = meta_beh_test.copy()
beh_test_probs["behaviour_prob"] = Pb_test

aca_test_probs = meta_aca_test.copy()
aca_test_probs["academic_prob"] = Pa_test
aca_test_probs["actual"] = y_aca_test.values

stack_test_df = beh_test_probs.merge(
    aca_test_probs,
    on=["student_key", "course_key", "academic_period_key", "snapshot_date"],
    how="inner"
)

stack_X_test = stack_test_df[["behaviour_prob", "academic_prob"]]
stack_y_test = stack_test_df["actual"]

# evaluate stacking on test set
stack_prob = meta_model.predict_proba(stack_X_test)[:, 1]
stack_pred = (stack_prob >= 0.5).astype(int)

print(confusion_matrix(stack_y_test, stack_pred))
print(classification_report(stack_y_test, stack_pred, zero_division=0))
print("Stacking Test ROC-AUC:", roc_auc_score(stack_y_test, stack_prob))

[[11  3]
 [ 3 17]]
              precision    recall  f1-score   support

           0       0.79      0.79      0.79        14
           1       0.85      0.85      0.85        20

    accuracy                           0.82        34
   macro avg       0.82      0.82      0.82        34
weighted avg       0.82      0.82      0.82        34

Stacking Test ROC-AUC: 0.8607142857142858


In [39]:
print(meta_model.coef_)
print(meta_model.intercept_)

[[1.17780096 1.88115342]]
[-1.55526901]
